<a href="https://colab.research.google.com/github/TrSaleMane/municipality-ai-course/blob/main/docs/day1_llm_basic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏛️ Day 1：LLM API 基礎体験
## 新人SE向け 自治体AIシステム開発 模擬体験講座

---

### このノートブックの目標

1. LLM（大規模言語モデル）のAPIを呼び出す方法を理解する
2. プロンプトエンジニアリングの基本を体験する
3. 自治体窓口AIのプロトタイプを作る

### 使用するAPI（どちらか選択）
- **Google Gemini API**（無料枠あり ← 推奨）
- **OpenAI API**（有料）

> 📌 **参考：ガバメントAI「源内」** では Amazon Bedrock を使用していますが、  
> 本講座では無料で体験できる Gemini API を使います。  
> Copyright © デジタル庁 / Licensed under MIT License

---
## Section 1：環境セットアップ

In [ ]:
# 必要なライブラリをインストール
!pip install -q google-generativeai openai

In [ ]:
# ======================================================
# APIキーの設定
# Google Colab の「シークレット」機能を使う（推奨）
# 左サイドバーの鍵アイコン → GEMINI_API_KEY を登録
# ======================================================
from google.colab import userdata
import os

# Gemini API キー
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

# OpenAI を使う場合はこちら
# OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

print('✅ APIキーの読み込み完了')

---
## Section 2：LLMとはなにか

```
【あなた】
  「住民票を取得するにはどうすればいいですか？」
          ↓ プロンプト（テキスト）
【LLM API】
  大量のテキストで学習したモデルが応答を生成
          ↓ レスポンス（テキスト）
【あなた】
  「市区町村の窓口またはコンビニのマルチコピー機で...」
```

### システムプロンプトとユーザープロンプトの違い

| 種類 | 役割 | 例 |
|------|------|----|
| **システムプロンプト** | AIの役割・ルールを定義 | 「あなたは市役所の窓口担当者です」 |
| **ユーザープロンプト** | ユーザーの質問・指示 | 「住民票の取り方を教えて」 |

In [ ]:
# ======================================================
# 【演習 2-1】Gemini API で最初のチャット
# ======================================================
import google.generativeai as genai

genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel('gemini-1.5-flash')

# シンプルな質問
response = model.generate_content('自治体システムとはどのようなものですか？簡潔に教えてください。')
print(response.text)

---
## Section 3：プロンプトエンジニアリング入門

同じ質問でも、**プロンプトの書き方次第で回答の質が大きく変わります。**

### 良いプロンプトの要素
1. **役割（Role）** を与える → 「あなたは市役所の窓口担当者です」
2. **制約（Constraint）** を設ける → 「300文字以内で」「箇条書きで」
3. **文脈（Context）** を提供する → 「高齢者向けに説明してください」
4. **出力形式（Format）** を指定する → 「手順をステップ形式で」

In [ ]:
# ======================================================
# 【演習 3-1】プロンプトの改善比較
# ======================================================

def call_llm(prompt: str) -> str:
    response = model.generate_content(prompt)
    return response.text

# ❌ 悪いプロンプト例
bad_prompt = '住民票について教えて'

# ✅ 良いプロンプト例
good_prompt = """
あなたは日本の市役所の窓口担当者です。
住民から「住民票の写し」を取得する方法について質問されました。

以下の条件で回答してください：
- 丁寧で分かりやすい言葉を使う
- 取得方法を「窓口」「コンビニ」「郵便」の3つに分けて説明する
- 各方法の必要書類と費用も含める
- 高齢者にも理解しやすい表現にする
"""

print('=' * 50)
print('❌ 悪いプロンプトの結果：')
print('=' * 50)
print(call_llm(bad_prompt))

print('\n' + '=' * 50)
print('✅ 良いプロンプトの結果：')
print('=' * 50)
print(call_llm(good_prompt))

---
## Section 4：チャット履歴の管理

実際の窓口AIでは、**会話の流れ（コンテキスト）を保持**する必要があります。

```
住民：「住民票を取りたいです」
AI ：「窓口、コンビニ、郵便の3通りあります」
住民：「コンビニの場合はどうすればいいですか？」  ← 前の会話を踏まえた質問
AI ：「マイナンバーカードが必要です...」
```

In [ ]:
# ======================================================
# 【演習 4-1】チャット履歴を保持する会話
# ======================================================

class MunicipalityChatBot:
    """自治体窓口チャットボット（履歴管理あり）"""

    SYSTEM_PROMPT = """
    あなたは日本の市役所の窓口AIアシスタントです。
    住民からの質問に丁寧に答えてください。

    ルール：
    - 正確で分かりやすい情報を提供する
    - 不明な点は「窓口にお問い合わせください」と案内する
    - 個人情報（氏名・住所・マイナンバー等）は絶対に聞かない
    - 回答は300文字以内を目安にする
    """

    def __init__(self):
        self.chat = model.start_chat(history=[])
        # システムプロンプトで初期化
        self.chat.send_message(self.SYSTEM_PROMPT)
        self.history = []

    def ask(self, user_input: str) -> str:
        response = self.chat.send_message(user_input)
        answer = response.text
        self.history.append({'user': user_input, 'ai': answer})
        return answer

    def show_history(self):
        for i, turn in enumerate(self.history, 1):
            print(f'\n--- ターン {i} ---')
            print(f'住民: {turn["user"]}')
            print(f'AI : {turn["ai"]}')


# チャットボットを起動
bot = MunicipalityChatBot()

# 会話シミュレーション
questions = [
    '住民票の写しを取得したいのですが、どうすればいいですか？',
    'コンビニで取得する場合、何が必要ですか？',
    '手数料はいくらですか？'
]

for q in questions:
    print(f'\n💬 住民: {q}')
    answer = bot.ask(q)
    print(f'🏛️  AI: {answer}')

---
## Section 5：インタラクティブチャット体験

実際にキーボードから入力して、自治体窓口AIと会話してみましょう。

In [ ]:
# ======================================================
# 【演習 5-1】インタラクティブチャット
# 「quit」または「終了」と入力すると終了します
# ======================================================

bot2 = MunicipalityChatBot()

print('🏛️ 市役所窓口AIへようこそ')
print('ご質問をどうぞ（「終了」と入力するとチャットを終了します）')
print('-' * 50)

while True:
    user_input = input('あなた: ').strip()
    if not user_input:
        continue
    if user_input in ['quit', '終了', 'exit']:
        print('チャットを終了します。ありがとうございました。')
        break
    answer = bot2.ask(user_input)
    print(f'AI: {answer}\n')

---
## 🎯 チャレンジ課題

以下のうち、1つ以上に挑戦してみてください。

### 課題 A：システムプロンプトを改善する
`MunicipalityChatBot` の `SYSTEM_PROMPT` を書き換えて、
より自然で丁寧な回答ができるように改善してください。

### 課題 B：別の業務に対応させる
住民票以外の業務（例：「土木課への道路損傷報告」「保育所の申込み」）に
特化したチャットボットを作ってください。

### 課題 C：回答の評価機能を追加する
各回答に対して「役に立った👍 / 役に立たなかった👎」のフィードバック機能を
実装してください。

---

## Day 1 まとめ

| 学んだこと | ポイント |
|-----------|--------|
| LLM APIの呼び出し | `generate_content()` でテキスト生成 |
| プロンプトエンジニアリング | 役割・制約・文脈・形式を明示する |
| チャット履歴の管理 | 会話の流れを保持することで自然な対話が可能 |
| 自治体AI特有のルール | 個人情報を聞かない・不明点は窓口へ案内 |

### 次のDay 2では...
実際の自治体FAQドキュメントを読み込んで、
**正確な情報に基づいて回答するRAGシステム**を構築します！